In [7]:
from factordb.factordb import FactorDB
from Crypto.Util.number import inverse, long_to_bytes

# Given RSA public key and ciphertext
n = 43941819371451617899582143885098799360907134939870946637129466519309346255747
e = 65537
c = 9002431156311360251224219512084136121048022631163334079215596223698721862766

# 1. Connect to FactorDB and factor n
f = FactorDB(n)
f.connect()
factors = f.get_factor_list()       # Gets [p, q]

p, q = factors                     # p and q from database

print("p =", p)
print("q =", q)

# 2. Compute phi(n)
phi_n = (p - 1) * (q - 1)
print("phi(n) =", phi_n)

# 3. Calculate the private key exponent d
d = inverse(e, phi_n)
print("Private exponent d =", d)

# 4. Decrypt ciphertext
plaintext_num = pow(c, d, n)
plaintext_bytes = long_to_bytes(plaintext_num)

# Print the decrypted text/flag
print("Decrypted plaintext:", plaintext_bytes.decode())

p = 205237461320000835821812139013267110933
q = 214102333408513040694153189550512987959
phi(n) = 43941819371451617899582143885098799360487795145142432760613501190745566156856
Private exponent d = 42863673506531127160266519316271436658935017712647978759376543290403486562425
Decrypted plaintext: THM{Psssss_4nd_Qsssssss}


In [20]:
import hmac
import hashlib

target_hash = "1484c3a5d65a55d70984b4d10b1884bda8876c1d"
message = b"CanYouGuessMySecret"

with open("rockyou.txt", "rb") as f:   # 🔥 WICHTIG: binary mode
    for line in f:
        key = line.rstrip(b"\r\n")     # 🔥 nur newline entfernen

        h = hmac.new(key, message, hashlib.sha1).hexdigest()

        if h == target_hash:
            print("Key gefunden:", key)
            print("Als String:", key.decode(errors="replace"))
            break

Key gefunden: b'sunshine'
Als String: sunshine


In [34]:
import requests
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad

# Configuration
url = "http://bcts.thm/labs/lab3/process.php"
encryption_key = b"1234567890123456"  # Must be 16 bytes (same as in the JavaScript)
wordlist_path = "rockyou.txt"        # Path to the wordlist

# Function to encrypt a message
def encrypt_message(message, iv):
    # Pad the message to a multiple of the block size (16 bytes for AES)
    padded_message = pad(message.encode(), AES.block_size)
    # Encrypt using AES-CBC
    cipher = AES.new(encryption_key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(padded_message)
    # Encode ciphertext and IV in Base64 for transmission
    return base64.b64encode(ciphertext).decode(), base64.b64encode(iv).decode()

# Function to send the payload
def send_payload(ciphertext, iv):
    payload = {"data": ciphertext, "iv": iv}
    response = requests.post(url, json=payload)
    return response.text

# Main bruteforce function
def bruteforce():
    with open(wordlist_path, "r") as f:
        words = f.readlines()

    for word in words:
        word = word.strip()
        print(f"Trying: {word}")
        # Generate a random IV (16 bytes)
        iv = AES.get_random_bytes(16)
        # Encrypt the current word
        ciphertext, iv_base64 = encrypt_message(word, iv)
        # Send the payload to the server
        response = send_payload(ciphertext, iv_base64)
        print(f"Response: {response}")
        # Check if the response indicates success
        if "Access granted!" in response:
            print(f"[+] Found the correct message: {word}")
            break

if __name__ == "__main__":
    bruteforce()

Trying: jadmxqtideg
Response: Message jadmxqtideg is invalid!
Trying: pyasosedg
Response: Message pyasosedg is invalid!
Trying: qdmicq
Response: Message qdmicq is invalid!
Trying: rvtmrledjot
Response: Message rvtmrledjot is invalid!
Trying: cchsdnrnt
Response: Message cchsdnrnt is invalid!
Trying: uordqsi
Response: Message uordqsi is invalid!
Trying: wqrtzyuzn
Response: Message wqrtzyuzn is invalid!
Trying: oqqtlujoobu
Response: Message oqqtlujoobu is invalid!
Trying: nfhjjecvspui
Response: Message nfhjjecvspui is invalid!
Trying: hxctsgqe
Response: Message hxctsgqe is invalid!
Trying: ggnmafthhfny
Response: Message ggnmafthhfny is invalid!
Trying: inhvxl
Response: Message inhvxl is invalid!
Trying: cwqbeztzrqkq
Response: Message cwqbeztzrqkq is invalid!
Trying: vjhshcfj
Response: Message vjhshcfj is invalid!
Trying: uoenskaerbjd
Response: Message uoenskaerbjd is invalid!
Trying: aufebt
Response: Message aufebt is invalid!
Trying: yilnjlayk
Response: Message yilnjlayk is invalid!
Tryi

In [33]:
import base64, sys
from binascii import unhexlify, hexlify

original_token = input("Enter the original token (hex string): ")

cipher_bytes = bytearray(unhexlify(original_token))

# AES block size
block_size = 16

# Debug: Print IV (first 16 bytes) before modification
print("\n[DEBUG] Original IV (First 16 Bytes):", hexlify(cipher_bytes[:block_size]).decode())

guest_offset = 0

xor_diff = [
    0x01,  # '0' -> '1'
]

# Apply bit flipping to the IV (first 16 bytes)
for i, diff in enumerate(xor_diff):
    print(f"[DEBUG] Modifying byte at offset {guest_offset + i}: {hex(cipher_bytes[guest_offset + i])} XOR {hex(diff)}")
    cipher_bytes[guest_offset + i] ^= diff

print("\n[DEBUG] Modified IV (First 16 Bytes):", hexlify(cipher_bytes[:block_size]).decode())

# Encode the modified token back to hex
modified_token = hexlify(cipher_bytes).decode()

print("\nModified Token:")
print(modified_token)
print("\nUse this token as the new 'role' cookie in your browser to log in as admin.")


[DEBUG] Original IV (First 16 Bytes): 5da102b52a6907e9b9a3e60510885689
[DEBUG] Modifying byte at offset 0: 0x5d XOR 0x1

[DEBUG] Modified IV (First 16 Bytes): 5ca102b52a6907e9b9a3e60510885689

Modified Token:
5ca102b52a6907e9b9a3e60510885689e41ed609def2f053079c38e65f5840dd48cbea747eac4a3c882eef0d20b3ec7b

Use this token as the new 'role' cookie in your browser to log in as admin.


Recap of Key Concepts

In this room, we’ve explored common cryptographic mistakes that developers often make, along with practical demonstrations of how these errors can be exploited. Let’s summarise the critical concepts and vulnerabilities you’ve encountered:

    Brute-forcing Keys:
        Weak or predictable keys, such as those derived from timestamps or short alphanumeric strings, can be cracked using brute-force attacks with tools like Hashcat or .
    Breaking Hashes:
        Outdated hash functions like and SHA-1 are vulnerable to attacks such as collision and preimage attacks.
        Lack of salting enables attackers to use rainbow tables to reverse hashes into plaintext.
    Keys Exposed in Client-Side Code:
        Hardcoding encryption keys or secrets in front-end code exposes them to anyone with access to the application.
        This allows attackers to decrypt sensitive data or impersonate users.
    Bit Flipping Attacks:
        Unauthenticated encryption (e.g., -CBC without a MAC) enables attackers to modify ciphertext, resulting in controlled changes to decrypted plaintext.

Best Practices for Avoiding Cryptographic Mistakes

To secure cryptographic implementations and prevent the vulnerabilities explored in this room, follow these best practices:

    Use Strong Keys and Secure Algorithms
        Generate keys with sufficient and length (e.g., -256 for symmetric encryption).
        Use modern, secure algorithms such as -GCM, -2048, and .
        Regularly update cryptographic libraries to protect against newly discovered vulnerabilities.
    Avoid Exposing Keys in Client-Side Code
        Never hardcode encryption keys, secrets, or sensitive credentials in JavaScript or other client-side files.
        Store secrets securely on the server side and use environment variables or key management systems like KMS or Azure Key Vault.
    Implement Authenticated Encryption
        Always pair encryption with checks using authenticated encryption modes like -GCM or -CCM.
        Ensure that any data transmitted over untrusted channels is encrypted and includes protection.
    Secure Implementations
        Use larger public exponents (e = 65537) and ensure that encrypted messages include padding (e.g., OAEP or PKCS#1).
        Avoid encrypting the same plaintext for multiple recipients with the same public exponent.
    Educate Developers
        Ensure that development teams understand secure cryptographic practices and the risks of misconfigurations.
        Conduct regular training and code reviews to identify and fix potential issues before deployment.


https://github.com/iagox86/hash_extender

hashcat -m 1400 -a 3 -o hashcat_output.txt --hex-salt --hex-charset=0123456789abcdef --hex-wordlist=rockyou.txt
hashcat auch cooles tool für Passwort-Cracking

sha-256 hash extension attack tool

In [ ]:
import hmac
import hashlib

key = b'supersecretkey'
message = b'important_data'

hmac_hash = hmac.new(key, message, hashlib.sha256).hexdigest()
print(hmac_hash)

In this room, we've discussed Length Extension Attacks and explored how certain hash functions, like SHA256, can be vulnerable if not handled properly.
Key Takeaways

    Understanding Hash Functions: You now know how cryptographic hash functions work and why properties like pre-image resistance and collision resistance are crucial for securing data.

    Exploiting Vulnerabilities: We've seen how attackers can modify hashed data and generate a valid signature without knowing the original input, thanks to the weaknesses in how some hashes process data.

    Real-World Impact: From forging signatures to altering signed cookies, we covered how these attacks play out in real life and the risks they pose if your system relies on vulnerable hashing algorithms.

    Mitigation Strategies: The best defence against length extension attacks is using HMAC with secure hash functions like SHA-256, coupled with good key management and encryption practices.
